# Music Annotation Project — Notebook 33
## V8.2d Distance-Two Semantic Review Tool

**Runtime:** CPU is sufficient.

This notebook is a development-only Gradio reviewer for the candidates produced by Notebook 32. It does **not** retrain, refit, recalibrate, relabel the V8.2 source run, change thresholds, or generate replacement corrected pages.

It merges two complementary Notebook 32 queues:

1. `distance2_annotation_high_pprint.csv` — current ANNOTATION truth, exactly two Chebyshev pixels from clean ink, with comparatively high V8.2 `p_print`;
2. `distance2_annotation_review_queue.csv` — morphology-ranked distance-two cases, including thin-continuous-proxy candidates.

The reviewer creates a de-duplicated, page-diversified sequence with:

- a **core batch of up to 100** cases;
- an **extension batch of up to 100** additional cases.

Complete the core batch first. The extension batch is available if the first 100 reviews do not provide sufficiently stable or diverse evidence.

### Review categories

- Staff line
- Hairpin
- Stem
- Slur
- Text glyph
- Other printed structure
- Genuine annotation
- Registration ambiguity
- Uncertain

Every decision is written immediately to CSV and Excel. Existing reviews are resumed rather than overwritten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gradio pandas numpy opencv-python-headless pillow openpyxl pyarrow

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from functools import lru_cache
import json, hashlib, math, os, platform, shutil, sys, threading

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import gradio as gr

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82C_ROOT = ROOT / 'observed_print_classifier_v8_2c_label_sensitivity'
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v8_2d_semantic_review'
SOURCE_V82C_RUN_OVERRIDE = None
REVIEW_RUN_OVERRIDE = None  # Set to an existing Notebook 33 run directory to resume it.

CORE_REVIEW_TARGET = 100
EXTENSION_REVIEW_TARGET = 100
TOTAL_REVIEW_TARGET = CORE_REVIEW_TARGET + EXTENSION_REVIEW_TARGET

CROP_RADIUS = 55
DISPLAY_SCALE = 3
PUBLIC_LINK = True
REVIEWER_NAME = 'David Bolton'

SUPPORTED = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
CATEGORIES = [
    'Staff line',
    'Hairpin',
    'Stem',
    'Slur',
    'Text glyph',
    'Other printed structure',
    'Genuine annotation',
    'Registration ambiguity',
    'Uncertain',
]

if REVIEW_RUN_OVERRIDE is None:
    RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_2d_semantic_review'
    OUT = OUTPUT_ROOT / RUN
    RESUMING_REVIEW = False
else:
    OUT = Path(REVIEW_RUN_OVERRIDE)
    RUN = OUT.name
    RESUMING_REVIEW = True

TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
SNAPSHOTS = OUT / 'review_snapshots'
for path in [OUT, TABLES, AUDIT, SNAPSHOTS]:
    path.mkdir(parents=True, exist_ok=RESUMING_REVIEW)

REVIEW_CSV = TABLES / 'semantic_review.csv'
REVIEW_XLSX = TABLES / 'semantic_review.xlsx'
SUMMARY_CSV = TABLES / 'semantic_review_summary.csv'
SUMMARY_XLSX = TABLES / 'semantic_review_summary.xlsx'
SAMPLING_CSV = TABLES / 'semantic_review_sampling_manifest.csv'
IDENTITY_JSON = OUT / 'experiment_identity.json'

LOCK = threading.RLock()
print('Output:', OUT)

## 2. Resolve the latest complete Notebook 32 run and load both queues

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def image_map(folder):
    folder = Path(folder)
    return {
        p.stem: p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED
    } if folder.exists() else {}


REQUIRED_V82C = [
    'experiment_identity.json',
    'tables/distance2_annotation_high_pprint.csv',
    'tables/distance2_annotation_review_queue.csv',
]


def complete_runs(root):
    hits = []
    root = Path(root)
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p / rel).exists() for rel in REQUIRED_V82C):
                hits.append(p)
    return sorted(hits, key=lambda p: p.stat().st_mtime)


if SOURCE_V82C_RUN_OVERRIDE is not None:
    SOURCE = Path(SOURCE_V82C_RUN_OVERRIDE)
    missing = [rel for rel in REQUIRED_V82C if not (SOURCE / rel).exists()]
    if missing:
        raise RuntimeError(f'Incomplete Notebook 32 source override: {missing}')
else:
    runs = complete_runs(V82C_ROOT)
    if not runs:
        raise RuntimeError(f'No complete Notebook 32 run found below {V82C_ROOT}')
    SOURCE = runs[-1]

SOURCE_IDENTITY = json.loads((SOURCE / 'experiment_identity.json').read_text())
if SOURCE_IDENTITY.get('version') != '8.2c-whole-population-label-sensitivity-audit':
    raise RuntimeError(f"Unexpected Notebook 32 version: {SOURCE_IDENTITY.get('version')}")

HIGH_PATH = SOURCE / 'tables' / 'distance2_annotation_high_pprint.csv'
MORPH_PATH = SOURCE / 'tables' / 'distance2_annotation_review_queue.csv'
high = pd.read_csv(HIGH_PATH)
morph = pd.read_csv(MORPH_PATH)

required_common = {'page','x','y','p_print'}
for label, frame in [('high-probability', high), ('morphology-ranked', morph)]:
    missing = required_common - set(frame.columns)
    if missing:
        raise RuntimeError(f'{label} queue missing columns: {sorted(missing)}')
    frame['page'] = frame.page.astype(str)
    frame['x'] = pd.to_numeric(frame.x, errors='raise').astype(int)
    frame['y'] = pd.to_numeric(frame.y, errors='raise').astype(int)
    frame['p_print'] = pd.to_numeric(frame.p_print, errors='raise').astype(float)
    if frame.duplicated(['page','y','x']).any():
        raise RuntimeError(f'{label} queue contains duplicate page/y/x rows')

CLEAN_MAP = image_map(ROOT / 'source' / 'clean')
OBSERVED_MAP = image_map(ROOT / 'aligned')
all_pages = sorted(set(high.page) | set(morph.page))
missing_clean = sorted(set(all_pages) - set(CLEAN_MAP))
missing_observed = sorted(set(all_pages) - set(OBSERVED_MAP))
if missing_clean or missing_observed:
    raise RuntimeError({
        'missing_clean_pages': missing_clean,
        'missing_observed_pages': missing_observed,
    })

print('Notebook 32 source:', SOURCE)
print('High-probability rows:', len(high))
print('Morphology-ranked rows:', len(morph))

## 3. Merge, de-duplicate and page-diversify the review sequence

In [ ]:
def candidate_id(page, y, x):
    return hashlib.sha256(f'{page}:{int(y)}:{int(x)}'.encode()).hexdigest()[:20]


def page_diversified_order(frame, score_column=None, descending=True):
    q = frame.copy()
    q['_input_rank'] = np.arange(len(q))
    if score_column and score_column in q.columns:
        q = q.sort_values(score_column, ascending=not descending, kind='stable')
    else:
        q = q.sort_values('_input_rank', kind='stable')

    groups = {page: g.to_dict('records') for page, g in q.groupby('page', sort=False)}
    if score_column and score_column in q.columns:
        page_order = sorted(
            groups,
            key=lambda p: groups[p][0].get(score_column, -np.inf),
            reverse=descending,
        )
    else:
        page_order = list(groups)

    out = []
    cursor = {p: 0 for p in page_order}
    while True:
        added = False
        for p in page_order:
            i = cursor[p]
            if i < len(groups[p]):
                out.append(groups[p][i])
                cursor[p] += 1
                added = True
        if not added:
            break
    return out


high = high.copy()
morph = morph.copy()
high['candidate_id'] = [candidate_id(p,y,x) for p,y,x in zip(high.page, high.y, high.x)]
morph['candidate_id'] = [candidate_id(p,y,x) for p,y,x in zip(morph.page, morph.y, morph.x)]

high_ids = set(high.candidate_id)
morph_ids = set(morph.candidate_id)

high_order = page_diversified_order(high, 'p_print', descending=True)
# Notebook 32 already wrote the morphology queue in rank order.
morph_order = page_diversified_order(morph, None)

# Full metadata union, preferring the morphology queue's richer fields where present.
union = pd.concat([high, morph], ignore_index=True, sort=False)
union = union.sort_values(['candidate_id'], kind='stable')
union = union.groupby('candidate_id', as_index=False, sort=False).first()
union = union.set_index('candidate_id', drop=False)

sequence = []
seen = set()
i = j = 0
while len(sequence) < min(TOTAL_REVIEW_TARGET, len(union)) and (i < len(high_order) or j < len(morph_order)):
    if i < len(high_order):
        cid = high_order[i]['candidate_id']; i += 1
        if cid not in seen:
            sequence.append(cid); seen.add(cid)
    if len(sequence) >= min(TOTAL_REVIEW_TARGET, len(union)):
        break
    if j < len(morph_order):
        cid = morph_order[j]['candidate_id']; j += 1
        if cid not in seen:
            sequence.append(cid); seen.add(cid)

queue = union.loc[sequence].copy().reset_index(drop=True)
queue['queue_order'] = np.arange(1, len(queue) + 1)
queue['sampling_stage'] = np.where(
    queue.queue_order <= CORE_REVIEW_TARGET,
    'core_100',
    'extension_100',
)
queue['in_high_probability_queue'] = queue.candidate_id.isin(high_ids)
queue['in_morphology_queue'] = queue.candidate_id.isin(morph_ids)
queue['sampling_origin'] = np.select(
    [
        queue.in_high_probability_queue & queue.in_morphology_queue,
        queue.in_high_probability_queue,
        queue.in_morphology_queue,
    ],
    ['both','high_probability','morphology_ranked'],
    default='unknown',
)
queue['review_category'] = ''
queue['review_note'] = ''
queue['reviewer'] = ''
queue['reviewed_utc'] = ''
queue['review_revision'] = 0

queue.to_csv(SAMPLING_CSV, index=False)

# Initialise a new review table, or merge durable review fields from an explicitly resumed run.
reviews = queue.copy()
if RESUMING_REVIEW and REVIEW_CSV.exists():
    prior = pd.read_csv(REVIEW_CSV).fillna('')
    prior_fields = [
        'candidate_id','review_category','review_note','reviewer',
        'reviewed_utc','review_revision'
    ]
    missing_prior = set(prior_fields) - set(prior.columns)
    if missing_prior:
        raise RuntimeError(f'Resumed review CSV lacks fields: {sorted(missing_prior)}')
    reviews = reviews.drop(columns=[
        'review_category','review_note','reviewer','reviewed_utc','review_revision'
    ]).merge(prior[prior_fields], on='candidate_id', how='left', validate='one_to_one')
    for col in ['review_category','review_note','reviewer','reviewed_utc']:
        reviews[col] = reviews[col].fillna('')
    reviews['review_revision'] = pd.to_numeric(
        reviews['review_revision'], errors='coerce'
    ).fillna(0).astype(int)

reviews.to_csv(REVIEW_CSV, index=False)
reviews.to_excel(REVIEW_XLSX, index=False, engine='openpyxl')

print('Merged unique candidates:', len(union))
print('Review sequence:', len(queue))
print(queue[['sampling_stage','sampling_origin']].value_counts().sort_index())

## 4. Rendering and truth-map helpers

In [ ]:
ADAPTIVE_BLOCK_SIZE = 41
ADAPTIVE_C = 15
TRUTH_KERNEL_SIZE = 3
TRUTH_BGR = {0:(0,0,220), 1:(0,200,0), 2:(0,190,255)}


def printed_ink(clean):
    return cv2.adaptiveThreshold(
        clean, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C
    ) > 0


def truth_maps(clean):
    ink = printed_ink(clean)
    k = np.ones((TRUTH_KERNEL_SIZE, TRUTH_KERNEL_SIZE), np.uint8)
    near = cv2.dilate(ink.astype(np.uint8), k) > 0
    eroded = cv2.erode(ink.astype(np.uint8), k) > 0
    boundary = np.logical_xor(near, eroded)
    return ink, near, eroded, boundary


def crop_with_pad(img, y, x, radius=CROP_RADIUS, pad_value=255):
    h, w = img.shape[:2]
    y0, y1 = y-radius, y+radius+1
    x0, x1 = x-radius, x+radius+1
    top=max(0,-y0); left=max(0,-x0); bottom=max(0,y1-h); right=max(0,x1-w)
    y0=max(0,y0); x0=max(0,x0); y1=min(h,y1); x1=min(w,x1)
    crop = img[y0:y1, x0:x1]
    return cv2.copyMakeBorder(crop, top,bottom,left,right, cv2.BORDER_CONSTANT, value=pad_value)


@lru_cache(maxsize=12)
def page_context(page):
    obs = cv2.imread(str(OBSERVED_MAP[page]), cv2.IMREAD_GRAYSCALE)
    clean = cv2.imread(str(CLEAN_MAP[page]), cv2.IMREAD_GRAYSCALE)
    if obs is None or clean is None or obs.shape != clean.shape:
        raise RuntimeError(f'{page}: observed/clean read or shape failure')
    ink, near, eroded, boundary = truth_maps(clean)
    return obs, clean, ink, near, eroded, boundary


def registration_overlay(obs_crop, ink_crop, dx, dy):
    out = cv2.cvtColor(obs_crop, cv2.COLOR_GRAY2BGR)
    contour = cv2.morphologyEx(
        ink_crop.astype(np.uint8), cv2.MORPH_GRADIENT,
        np.ones((3,3), np.uint8)
    ) > 0
    out[contour] = (0,255,0)
    c = (CROP_RADIUS, CROP_RADIUS)
    n = (CROP_RADIUS + int(dx), CROP_RADIUS + int(dy))
    cv2.drawMarker(out, c, (255,0,255), cv2.MARKER_CROSS, 15, 1)
    if 0 <= n[0] < out.shape[1] and 0 <= n[1] < out.shape[0]:
        cv2.circle(out, n, 4, (255,255,0), 1)
        cv2.line(out, c, n, (255,255,0), 1)
    return out


def truth_colour_patch(near, boundary, y, x):
    label = np.full(near.shape, 0, np.uint8)
    label[near & ~boundary] = 1
    label[boundary] = 2
    patch = crop_with_pad(label, y, x, pad_value=0)
    out = np.full((*patch.shape,3), 235, np.uint8)
    for k,bgr in TRUTH_BGR.items():
        out[patch == k] = bgr
    cv2.drawMarker(out, (CROP_RADIUS,CROP_RADIUS), (255,0,255), cv2.MARKER_CROSS, 15, 1)
    return out


def make_review_image(row):
    page = str(row.page); x = int(row.x); y = int(row.y)
    obs, clean, ink, near, eroded, boundary = page_context(page)
    obs_crop = crop_with_pad(obs, y, x)
    clean_crop = crop_with_pad(clean, y, x)
    ink_crop = crop_with_pad(ink.astype(np.uint8), y, x, pad_value=0) > 0

    nx = int(row.get('nearest_clean_ink_x', x)) if pd.notna(row.get('nearest_clean_ink_x', np.nan)) else x
    ny = int(row.get('nearest_clean_ink_y', y)) if pd.notna(row.get('nearest_clean_ink_y', np.nan)) else y
    overlay = registration_overlay(obs_crop, ink_crop, nx-x, ny-y)
    truth = truth_colour_patch(near, boundary, y, x)

    panels = [
        cv2.cvtColor(obs_crop, cv2.COLOR_GRAY2BGR),
        cv2.cvtColor(clean_crop, cv2.COLOR_GRAY2BGR),
        overlay,
        truth,
    ]
    panels = [cv2.resize(p, (p.shape[1]*DISPLAY_SCALE, p.shape[0]*DISPLAY_SCALE), interpolation=cv2.INTER_NEAREST) for p in panels]
    panel_h, panel_w = panels[0].shape[:2]
    header = 76
    sheet = np.full((panel_h+header, panel_w*4, 3), 255, np.uint8)
    titles = ['Observed', 'Clean', 'Observed + clean contour', 'Derived truth']
    for i,(panel,title) in enumerate(zip(panels,titles)):
        sheet[header:, i*panel_w:(i+1)*panel_w] = panel
        cv2.putText(sheet, title, (i*panel_w+6, 21), cv2.FONT_HERSHEY_SIMPLEX, .46, (20,20,20), 1, cv2.LINE_AA)

    thin = bool(row.get('thin_continuous_proxy', False))
    caption1 = f"{page}  x={x} y={y}  p_print={float(row.p_print):.6f}  origin={row.sampling_origin}"
    caption2 = f"stage={row.sampling_stage}  thin_proxy={thin}  clean-distance=2  magenta=candidate  cyan=nearest clean ink"
    cv2.putText(sheet, caption1, (6, 46), cv2.FONT_HERSHEY_SIMPLEX, .46, (20,20,20), 1, cv2.LINE_AA)
    cv2.putText(sheet, caption2, (6, 68), cv2.FONT_HERSHEY_SIMPLEX, .42, (20,20,20), 1, cv2.LINE_AA)
    return cv2.cvtColor(sheet, cv2.COLOR_BGR2RGB)

## 5. Durable review persistence and summaries

In [ ]:
def make_summary(frame):
    reviewed = frame[frame.review_category.astype(str).str.len() > 0].copy()
    if reviewed.empty:
        return pd.DataFrame(columns=[
            'review_category','reviews','fraction','mean_p_print','median_p_print',
            'core_reviews','extension_reviews','high_probability_members','morphology_members'
        ])
    summary = reviewed.groupby('review_category', as_index=False).agg(
        reviews=('candidate_id','size'),
        mean_p_print=('p_print','mean'),
        median_p_print=('p_print','median'),
        core_reviews=('sampling_stage', lambda s: int((s == 'core_100').sum())),
        extension_reviews=('sampling_stage', lambda s: int((s == 'extension_100').sum())),
        high_probability_members=('in_high_probability_queue','sum'),
        morphology_members=('in_morphology_queue','sum'),
    )
    summary['fraction'] = summary.reviews / max(len(reviewed), 1)
    return summary.sort_values('reviews', ascending=False)


def atomic_write_reviews():
    global reviews
    tmp_csv = REVIEW_CSV.with_suffix('.csv.tmp')
    reviews.to_csv(tmp_csv, index=False)
    os.replace(tmp_csv, REVIEW_CSV)

    tmp_xlsx = REVIEW_XLSX.with_name(REVIEW_XLSX.stem + '.tmp.xlsx')
    reviews.to_excel(tmp_xlsx, index=False, engine='openpyxl')
    os.replace(tmp_xlsx, REVIEW_XLSX)

    summary = make_summary(reviews)
    tmp_sum_csv = SUMMARY_CSV.with_suffix('.csv.tmp')
    summary.to_csv(tmp_sum_csv, index=False)
    os.replace(tmp_sum_csv, SUMMARY_CSV)

    tmp_sum_xlsx = SUMMARY_XLSX.with_name(SUMMARY_XLSX.stem + '.tmp.xlsx')
    with pd.ExcelWriter(tmp_sum_xlsx, engine='openpyxl') as writer:
        summary.to_excel(writer, sheet_name='category_summary', index=False)
        reviews.to_excel(writer, sheet_name='all_reviews', index=False)
        reviews[reviews.review_category.astype(str).str.len() > 0].to_excel(writer, sheet_name='reviewed_only', index=False)
    os.replace(tmp_sum_xlsx, SUMMARY_XLSX)


# Persist the already initialised or resumed review table.
atomic_write_reviews()


def progress_text(frame):
    reviewed = frame.review_category.astype(str).str.len() > 0
    core = frame.sampling_stage.eq('core_100')
    extension = frame.sampling_stage.eq('extension_100')
    n = int(reviewed.sum())
    n_core = int((reviewed & core).sum())
    n_ext = int((reviewed & extension).sum())
    core_total = int(core.sum())
    ext_total = int(extension.sum())
    if n_core >= core_total and core_total > 0:
        status = 'Core review complete. Extension cases are available for additional evidence.'
    else:
        status = 'Complete the core review before interpreting the extension batch.'
    return (
        f"**Reviewed:** {n}/{len(frame)}  |  **Core:** {n_core}/{core_total}  |  "
        f"**Extension:** {n_ext}/{ext_total}  \n{status}"
    )

print(REVIEW_CSV)
print(REVIEW_XLSX)

## 6. Gradio reviewer

In [ ]:
FILTERS = [
    'All candidates',
    'Unreviewed only',
    'Reviewed only',
    'Core 100',
    'Extension 100',
    'High-probability queue',
    'Morphology-ranked queue',
    'Present in both queues',
]


def filtered_indices(filter_name):
    mask = pd.Series(True, index=reviews.index)
    if filter_name == 'Unreviewed only':
        mask &= reviews.review_category.astype(str).str.len().eq(0)
    elif filter_name == 'Reviewed only':
        mask &= reviews.review_category.astype(str).str.len().gt(0)
    elif filter_name == 'Core 100':
        mask &= reviews.sampling_stage.eq('core_100')
    elif filter_name == 'Extension 100':
        mask &= reviews.sampling_stage.eq('extension_100')
    elif filter_name == 'High-probability queue':
        mask &= reviews.in_high_probability_queue.astype(bool)
    elif filter_name == 'Morphology-ranked queue':
        mask &= reviews.in_morphology_queue.astype(bool)
    elif filter_name == 'Present in both queues':
        mask &= reviews.in_high_probability_queue.astype(bool) & reviews.in_morphology_queue.astype(bool)
    return reviews.index[mask].tolist()


def metadata_markdown(row, position, total):
    fields = [
        ('Record', f'{position+1}/{total}'),
        ('Queue order', row.queue_order),
        ('Stage', row.sampling_stage),
        ('Origin', row.sampling_origin),
        ('Page', row.page),
        ('Source group', row.get('source_group','')),
        ('Coordinate', f"({int(row.x)}, {int(row.y)})"),
        ('p_print', f"{float(row.p_print):.8f}"),
        ('Thin proxy', bool(row.get('thin_continuous_proxy', False))),
        ('Nearest clean distance', row.get('nearest_clean_ink_chebyshev', 2)),
        ('Current review', row.review_category or 'unreviewed'),
    ]
    return '\n'.join([f"- **{k}:** {v}" for k,v in fields])


def render(filter_name, position):
    ids = filtered_indices(filter_name)
    if not ids:
        blank = np.full((420, 1100, 3), 245, np.uint8)
        return blank, '**No records match this filter.**', progress_text(reviews), '', '', 0, ''
    position = int(max(0, min(int(position), len(ids)-1)))
    idx = ids[position]
    row = reviews.loc[idx]
    image = make_review_image(row)
    return (
        image,
        metadata_markdown(row, position, len(ids)),
        progress_text(reviews),
        str(row.review_category),
        str(row.review_note),
        position,
        str(row.candidate_id),
    )


def move(filter_name, position, delta):
    ids = filtered_indices(filter_name)
    if not ids:
        return render(filter_name, 0)
    new_pos = (int(position) + int(delta)) % len(ids)
    return render(filter_name, new_pos)


def jump(filter_name, requested):
    ids = filtered_indices(filter_name)
    if not ids:
        return render(filter_name, 0)
    pos = int(max(1, min(int(requested or 1), len(ids)))) - 1
    return render(filter_name, pos)


def save_review(candidate_id_value, category, note, filter_name, position, advance):
    global reviews
    if not candidate_id_value:
        return render(filter_name, position)
    if category not in CATEGORIES:
        return render(filter_name, position)
    with LOCK:
        hits = reviews.index[reviews.candidate_id.eq(candidate_id_value)].tolist()
        if len(hits) != 1:
            raise RuntimeError(f'Candidate ID does not resolve uniquely: {candidate_id_value}')
        idx = hits[0]
        reviews.at[idx, 'review_category'] = category
        reviews.at[idx, 'review_note'] = str(note or '')
        reviews.at[idx, 'reviewer'] = REVIEWER_NAME
        reviews.at[idx, 'reviewed_utc'] = datetime.now(timezone.utc).isoformat()
        reviews.at[idx, 'review_revision'] = int(reviews.at[idx, 'review_revision']) + 1
        atomic_write_reviews()
    return move(filter_name, position, 1 if advance else 0)


def clear_review(candidate_id_value, filter_name, position):
    global reviews
    if candidate_id_value:
        with LOCK:
            hits = reviews.index[reviews.candidate_id.eq(candidate_id_value)].tolist()
            if len(hits) == 1:
                idx = hits[0]
                for col in ['review_category','review_note','reviewer','reviewed_utc']:
                    reviews.at[idx, col] = ''
                reviews.at[idx, 'review_revision'] = int(reviews.at[idx, 'review_revision']) + 1
                atomic_write_reviews()
    return render(filter_name, position)


def export_files():
    with LOCK:
        atomic_write_reviews()
    return [str(REVIEW_CSV), str(REVIEW_XLSX), str(SUMMARY_CSV), str(SUMMARY_XLSX), str(SAMPLING_CSV)]


with gr.Blocks(title='Notebook 33 — V8.2d Semantic Reviewer') as demo:
    gr.Markdown('# Notebook 33 — V8.2d Distance-Two Semantic Reviewer')
    gr.Markdown('Choose one semantic category for each candidate. **Save & Next** writes immediately to CSV and Excel. Complete the core 100 first; the extension 100 is optional additional evidence.')

    progress = gr.Markdown(progress_text(reviews))
    with gr.Row():
        filter_box = gr.Dropdown(FILTERS, value='Unreviewed only', label='Queue filter')
        record_number = gr.Number(value=1, precision=0, label='Jump to record number in current filter')
        jump_btn = gr.Button('Jump')
        refresh_btn = gr.Button('↻ Refresh')

    review_image = gr.Image(label='Observed / clean / registration / derived truth', interactive=False, height=650)

    with gr.Row():
        metadata = gr.Markdown()
        with gr.Column():
            category = gr.Radio(CATEGORIES, label='Semantic review category')
            note = gr.Textbox(label='Optional note', lines=3, placeholder='Add a note only where it helps interpret the category.')
            advance_after_save = gr.Checkbox(value=True, label='Advance after saving')

    with gr.Row():
        prev_btn = gr.Button('◀ Previous')
        save_btn = gr.Button('Save & Next', variant='primary')
        save_stay_btn = gr.Button('Save current')
        clear_btn = gr.Button('Clear current review')
        next_btn = gr.Button('Next ▶')

    current_position = gr.State(0)
    current_candidate_id = gr.State('')

    with gr.Row():
        export_btn = gr.Button('Export / refresh review files')
        export_output = gr.Files(label='Review spreadsheets and sampling manifest')

    outputs = [review_image, metadata, progress, category, note, current_position, current_candidate_id]

    demo.load(lambda: render('Unreviewed only', 0), outputs=outputs)
    filter_box.change(lambda f: render(f, 0), inputs=filter_box, outputs=outputs)
    refresh_btn.click(render, inputs=[filter_box, current_position], outputs=outputs)
    jump_btn.click(jump, inputs=[filter_box, record_number], outputs=outputs)
    prev_btn.click(lambda f,p: move(f,p,-1), inputs=[filter_box,current_position], outputs=outputs)
    next_btn.click(lambda f,p: move(f,p,1), inputs=[filter_box,current_position], outputs=outputs)

    save_btn.click(
        lambda cid,cat,n,f,p,a: save_review(cid,cat,n,f,p,a),
        inputs=[current_candidate_id,category,note,filter_box,current_position,advance_after_save],
        outputs=outputs,
    )
    save_stay_btn.click(
        lambda cid,cat,n,f,p: save_review(cid,cat,n,f,p,False),
        inputs=[current_candidate_id,category,note,filter_box,current_position],
        outputs=outputs,
    )
    clear_btn.click(clear_review, inputs=[current_candidate_id,filter_box,current_position], outputs=outputs)
    export_btn.click(export_files, outputs=export_output)

print('Gradio version:', gr.__version__)

## 7. Write the reviewer identity and launch

In [ ]:
identity = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'version': '8.2d-distance-two-semantic-review',
    'resuming_review_run': bool(RESUMING_REVIEW),
    'development_only': True,
    'promotion_authority': False,
    'retrained': False,
    'refitted': False,
    'recalibrated': False,
    'source_notebook32_run': str(SOURCE),
    'source_experiment_identity_sha256': sha256_file(SOURCE / 'experiment_identity.json'),
    'high_probability_queue': str(HIGH_PATH),
    'high_probability_queue_sha256': sha256_file(HIGH_PATH),
    'morphology_queue': str(MORPH_PATH),
    'morphology_queue_sha256': sha256_file(MORPH_PATH),
    'review_targets': {
        'core': CORE_REVIEW_TARGET,
        'extension': EXTENSION_REVIEW_TARGET,
        'actual_queue_rows': len(reviews),
    },
    'categories': CATEGORIES,
    'sampling': {
        'method': 'alternating de-duplicated page-diversified high-probability and morphology-ranked queues',
        'core_interpretation': 'primary review batch',
        'extension_interpretation': 'optional additional evidence; not mandatory before examining core results',
    },
    'outputs': {
        'semantic_review_csv': str(REVIEW_CSV),
        'semantic_review_xlsx': str(REVIEW_XLSX),
        'summary_csv': str(SUMMARY_CSV),
        'summary_xlsx': str(SUMMARY_XLSX),
        'sampling_manifest': str(SAMPLING_CSV),
    },
    'software': {
        'python': sys.version,
        'opencv': cv2.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'gradio': gr.__version__,
        'platform': platform.platform(),
    },
}
IDENTITY_JSON.write_text(json.dumps(identity, indent=2, default=str))
print(json.dumps(identity, indent=2, default=str))

demo.queue(default_concurrency_limit=1).launch(
    share=PUBLIC_LINK,
    debug=True,
)

## Outputs to bring back

After the core 100 reviews, start with:

- `tables/semantic_review.csv`
- `tables/semantic_review_summary.csv`
- `tables/semantic_review_sampling_manifest.csv`
- `experiment_identity.json`

You may stop after the core 100 and inspect the category distribution before completing the extension 100. The extension exists to test whether the first-batch conclusions remain stable and to increase representation of candidates unique to either source queue.